# 03 — Backtest Results

Full backtest analysis: tearsheet, walk-forward results, in-sample vs OOS comparison.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.data.universe import UniverseProvider
from src.features.registry import build_features
from src.signals.combiner import SignalCombiner
from src.portfolio.constructor import PortfolioConstructor
from src.backtest.engine import BacktestEngine
from src.risk.manager import RiskManager
from src.analytics.performance import compute_metrics
from src.analytics.report import generate_report, generate_comparison_table
from src.visualization.tearsheet import create_tearsheet
from src.visualization.plots import (
    plot_equity_curve, plot_rolling_sharpe, plot_cost_sensitivity
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load and prepare data
cfg = load_config('../config.yaml')
universe = UniverseProvider(cfg.data)
tickers = universe.get_tickers()

loader = DataLoader(cfg.data)
raw_data = loader.load_universe(tickers)
prices = loader.build_price_matrix(raw_data)
volumes = loader.build_volume_matrix(raw_data)

valid_tickers = universe.filter_universe(prices, volumes)
prices = prices[valid_tickers]
volumes = volumes[valid_tickers]

cleaner = DataCleaner(cfg.data)
prices, returns = cleaner.clean(prices, volumes)

In [ ]:
# Compute signals
features = build_features(cfg.features)
signal_dict = {}
for feature in features:
    signal_dict[feature.name] = feature.compute(prices, returns)

combiner = SignalCombiner(cfg.signals)
combined = combiner.combine(signal_dict, returns)

In [ ]:
# Portfolio construction
constructor = PortfolioConstructor(cfg.portfolio)
weights = constructor.construct(combined, returns)

# Risk management
risk_mgr = RiskManager(cfg.risk)
adjusted = risk_mgr.apply(weights, returns)

# Backtest
engine = BacktestEngine(cfg.backtest, cfg.execution)
result = engine.run(adjusted, returns)

# Second pass with drawdown circuit breaker
adjusted2 = risk_mgr.apply(weights, returns, result.net_returns)
result = engine.run(adjusted2, returns)

In [ ]:
# Full performance report
report = generate_report(result, risk_free_rate=cfg.analytics.risk_free_rate)
print(report)

In [ ]:
# Strategy tearsheet
create_tearsheet(result, risk_free_rate=cfg.analytics.risk_free_rate,
                 title='Momentum/MR Strategy — Full Sample')
plt.show()

In [ ]:
# Equity curve with benchmark
plot_equity_curve(result, title='Strategy vs Benchmark')
plt.show()

In [ ]:
# Rolling Sharpe ratio
plot_rolling_sharpe(result.net_returns, risk_free_rate=cfg.analytics.risk_free_rate)
plt.show()

In [ ]:
# Cost sensitivity analysis
from src.config import ExecutionConfig
import dataclasses

cost_scenarios = {}
for cost_mult, label in [(0, '0 bps (gross)'), (0.5, '5 bps'), (1.0, '10 bps (base)'), (2.0, '20 bps')]:
    exec_cfg = dataclasses.replace(
        cfg.execution,
        fixed_cost_bps=cfg.execution.fixed_cost_bps * cost_mult,
        market_impact_bps=cfg.execution.market_impact_bps * cost_mult,
    )
    eng = BacktestEngine(cfg.backtest, exec_cfg)
    res = eng.run(adjusted2, returns)
    cost_scenarios[label] = res.net_returns

plot_cost_sensitivity(cost_scenarios)
plt.show()

In [ ]:
# Regime analysis: performance in different market regimes
market_return = returns.mean(axis=1)
market_vol = market_return.rolling(63).std() * np.sqrt(252)

# Define regimes
high_vol = market_vol > market_vol.quantile(0.75)
low_vol = market_vol < market_vol.quantile(0.25)
bull = market_return.rolling(252).mean() > 0
bear = ~bull

regime_metrics = {}
for regime_name, mask in [('Full Sample', pd.Series(True, index=result.net_returns.index)),
                           ('High Vol', high_vol), ('Low Vol', low_vol),
                           ('Bull', bull), ('Bear', bear)]:
    aligned = mask.reindex(result.net_returns.index).fillna(False)
    regime_ret = result.net_returns[aligned]
    if len(regime_ret) > 21:
        m = compute_metrics(regime_ret)
        regime_metrics[regime_name] = {
            'N Days': len(regime_ret),
            'Ann Return': f'{m.annual_return:.2%}',
            'Ann Vol': f'{m.annual_volatility:.2%}',
            'Sharpe': f'{m.sharpe_ratio:.2f}',
            'Max DD': f'{m.max_drawdown:.2%}',
        }

print(pd.DataFrame(regime_metrics).T.to_string())